In [1]:
# Import python modules
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Determine the absolute path to the src directory (one level up from notebooks)
module_path = os.path.abspath(os.path.join("..", "src"))
if module_path not in sys.path:
    sys.path.append(module_path)

project_root = os.path.abspath(os.path.join(".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
# Now this works because Python sees `src` as a top-level module
from src import analytics, plotting, utils

In [3]:
import yaml

In [4]:
# TODO: Read both no_bat and bat results. Make a copy of the config with co2-prices

In [5]:
BASE_FOLDER = os.path.dirname(os.getcwd())
RUNS_FOLDER = os.path.join(BASE_FOLDER, "runs")
BATCH_RUNS_FOLDER = os.path.join(RUNS_FOLDER, "batch_runs")

DATA_FOLDER = os.path.join(BASE_FOLDER, "data")

In [14]:
FOLDER = r"C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\runs\batch_runs\batch_128_4_4_4core_bat_and_no_bat\1_May16_Fri_h23_m07_s45-GTSEP_stochastic_v1-128_ES_PT"

In [15]:
decision_variables_folder = os.path.join(FOLDER, "decision_variables")
model_info_folder = os.path.join(FOLDER, "model_info")
dual_variables_folder = os.path.join(FOLDER, "dual_variables")

In [17]:
RESULTS_FOLDER = os.path.join(FOLDER, "results")

In [19]:
model_info = pd.read_csv(os.path.join(model_info_folder, "model_info.csv"))
config = yaml.safe_load(open(os.path.join(model_info_folder, "config.yaml")))
jsons = utils.read_jsons_from_dir(model_info_folder)

In [22]:
input_data_folder = os.path.join(
    os.path.dirname(os.getcwd()), "data", "processed", config["data_folder_name"]
)
input_data = utils.load_multi_year_csv_files_with_week_from_folder(
    years=config["years"], data_folder_path=input_data_folder
)

In [23]:
input_data.keys()

dict_keys(['batteries', 'branches', 'capacity_factors', 'generators', 'generator_costs', 'hourly_demand', 'nodes'])

In [24]:
hourly_demand = input_data["hourly_demand"]

In [25]:
hourly_demand

ES1 0        ES1 1        ES1 2        ES1 3        ES1 4  \
week hour                                                                    
1    0     1540.497187  3541.798151  1849.109695  2953.965919  2331.873044   
     1     1433.747317  3296.366678  1720.974297  2749.268707  2170.284209   
     2     1335.043919  3069.435065  1602.497346  2560.000934  2020.875437   
     3     1273.019026  2926.831980  1528.046816  2441.065682  1926.987452   
     4     1250.287740  2874.569876  1500.761703  2397.477519  1892.578775   
...                ...          ...          ...          ...          ...   
     187   2047.626125  4707.751813  2457.833322  3926.406254  3099.521508   
     188   1925.453849  4426.862275  2311.185901  3692.135952  2914.587553   
     189   1773.509624  4077.523257  2128.802225  3400.776730  2684.587365   
     190   1697.939835  3903.778741  2038.093309  3255.868591  2570.196274   
     191   1622.973531  3731.421714  1948.108777  3112.117659  2456.718688   

                 ES1 5        ES1 6        ES1 7        ES1 8        PT1 0  \
week hour                                                                    
1    0     3518.575677  2092.739520  1871.996365  2787.386472  2275.245735   
     1     3274.753423  1947.721616  1742.275019  2594.232504  2182.798908   
     2     3049.309730  1813.634710  1622.331663  2415.637880  2041.494173   
     3     2907.641648  1729.374936  1546.959650  2303.409601  1916.954406   
     4     2855.722211  1698.494867  1519.336825  2262.279453  1845.104541   
...                ...          ...          ...          ...          ...   
     187   4676.884541  2781.665653  2488.254244  3704.989148  3105.830178   
     188   4397.836710  2615.696670  2339.791748  3483.929770  2811.245730   
     189   4050.788199  2409.282996  2155.150686  3209.000818  2655.092023   
     190   3878.182872  2306.622709  2063.319055  3072.264309  2582.284159   
     191   3706.955937  2204.782246  1972.220772  2936.619751  2461.576385   

                 PT1 1  
week hour               
1    0     2474.754265  
     1     2374.201092  
     2     2220.505827  
     3     2085.045594  
     4     2006.895459  
...                ...  
     187   3378.169822  
     188   3057.754270  
     189   2887.907977  
     190   2808.715841  
     191   2677.423615  

[8760 rows x 11 columns]

In [26]:
import pandas as pd

# Assume hourly_demand is your dataframe and MultiIndex is ["week", "hour"]

# Sum across hours to get total demand for each week (sum across all nodes)
weekly_total = hourly_demand.sum(axis=1).groupby("week").sum()

In [27]:
def week_to_season(week):
    if 1 <= week <= 13:
        return "Winter"
    elif 14 <= week <= 26:
        return "Spring"
    elif 27 <= week <= 39:
        return "Summer"
    elif 40 <= week <= 52:
        return "Autumn"
    else:
        return "Unknown"


weekly_total = weekly_total.to_frame("total_demand")
weekly_total["season"] = weekly_total.index.map(week_to_season)

In [28]:
result = []
for season in ["Winter", "Spring", "Summer", "Autumn"]:
    season_weeks = weekly_total[weekly_total["season"] == season]
    if not season_weeks.empty:
        highest = season_weeks["total_demand"].idxmax()
        lowest = season_weeks["total_demand"].idxmin()
        result.append(
            {
                "season": season,
                "highest_week": highest,
                "highest_demand": season_weeks.loc[highest, "total_demand"],
                "lowest_week": lowest,
                "lowest_demand": season_weeks.loc[lowest, "total_demand"],
            }
        )

# Convert to DataFrame for pretty display
seasonal_extremes = pd.DataFrame(result)
print(seasonal_extremes)

   season  highest_week  highest_demand  lowest_week  lowest_demand
0  Winter             9    6.201472e+06           13   5.133926e+06
1  Spring            15    5.454663e+06           18   5.115533e+06
2  Summer            28    5.865407e+06           33   5.286365e+06
3  Autumn            48    6.217299e+06           44   5.136201e+06


In [29]:
# 1. Aggregate total demand by week
weekly_total = hourly_demand.sum(axis=1).groupby("week").sum()


# 2. Assign seasons
def week_to_season(week):
    if 1 <= week <= 13:
        return "Winter"
    elif 14 <= week <= 26:
        return "Spring"
    elif 27 <= week <= 39:
        return "Summer"
    elif 40 <= week <= 52:
        return "Autumn"
    else:
        return "Unknown"


weekly_total = weekly_total.to_frame("total_demand")
weekly_total["season"] = weekly_total.index.map(week_to_season)

# 3. Sort weeks in each season by demand descending
for season in ["Winter", "Spring", "Summer", "Autumn"]:
    season_weeks = weekly_total[weekly_total["season"] == season]
    sorted_weeks = season_weeks.sort_values("total_demand", ascending=False)
    print(f"\nSeason: {season}")
    print(sorted_weeks[["total_demand"]])


Season: Winter
      total_demand
week              
9     6.201472e+06
4     6.169147e+06
3     6.154159e+06
1     6.152327e+06
6     6.090839e+06
7     5.999782e+06
2     5.986252e+06
8     5.964497e+06
5     5.921483e+06
11    5.909982e+06
10    5.855242e+06
12    5.702891e+06
13    5.133926e+06

Season: Spring
      total_demand
week              
15    5.454663e+06
24    5.409832e+06
26    5.393359e+06
14    5.379059e+06
25    5.298752e+06
23    5.235780e+06
21    5.218462e+06
16    5.209588e+06
17    5.204575e+06
20    5.203051e+06
19    5.200285e+06
22    5.189140e+06
18    5.115533e+06

Season: Summer
      total_demand
week              
28    5.865407e+06
29    5.739834e+06
30    5.716870e+06
27    5.692057e+06
31    5.577737e+06
34    5.544469e+06
36    5.495223e+06
38    5.489905e+06
39    5.445019e+06
32    5.429892e+06
37    5.376055e+06
35    5.338719e+06
33    5.286365e+06

Season: Autumn
      total_demand
week              
48    6.217299e+06
50    6.131717e+06
49   

In [31]:
# 1. Aggregate total demand by week
weekly_total = hourly_demand.sum(axis=1).groupby("week").sum()


# 2. Assign seasons
def week_to_season(week):
    if 1 <= week <= 13:
        return "Winter"
    elif 14 <= week <= 26:
        return "Spring"
    elif 27 <= week <= 39:
        return "Summer"
    elif 40 <= week <= 52:
        return "Autumn"
    else:
        return "Unknown"


weekly_total = weekly_total.to_frame("total_demand")
weekly_total["season"] = weekly_total.index.map(week_to_season)

# 3. Calculate extremes and summary for each season
summary_rows = []
for season in ["Winter", "Spring", "Summer", "Autumn"]:
    season_weeks = weekly_total[weekly_total["season"] == season]
    if season_weeks.empty:
        continue
    highest_week = season_weeks["total_demand"].idxmax()
    highest_val = season_weeks.loc[highest_week, "total_demand"]
    lowest_week = season_weeks["total_demand"].idxmin()
    lowest_val = season_weeks.loc[lowest_week, "total_demand"]
    mean_val = season_weeks["total_demand"].mean()
    diff = highest_val - lowest_val
    pct_diff = 100 * diff / lowest_val if lowest_val != 0 else float("inf")
    summary_rows.append(
        {
            "season": season,
            "highest_week": highest_week,
            "highest_demand": highest_val,
            "lowest_week": lowest_week,
            "lowest_demand": lowest_val,
            "mean_weekly_demand": mean_val,
            "difference": diff,
            "percent_difference": pct_diff,
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df

,season,highest_week,highest_demand,lowest_week,lowest_demand,mean_weekly_demand,difference,percent_difference
0,Winter,9,6.201472e+06,13,5.133926e+06,5.941692e+06,1.067545e+06,20.793937
1,Spring,15,5.454663e+06,18,5.115533e+06,5.270160e+06,3.391295e+05,6.629406
2,Summer,28,5.865407e+06,33,5.286365e+06,5.538273e+06,5.790428e+05,10.953516
3,Autumn,48,6.217299e+06,44,5.136201e+06,5.578973e+06,1.081098e+06,21.048596


In [32]:
generators = input_data["generators"]

In [38]:
G = generators.index.get_level_values("generator").unique().tolist()

In [39]:
G

['ES1 0 CCGT',
 'ES1 0 coal',
 'ES1 0 offwind-ac',
 'ES1 0 onwind',
 'ES1 0 solar',
 'ES1 1 CCGT',
 'ES1 1 coal',
 'ES1 1 offwind-ac',
 'ES1 1 onwind',
 'ES1 1 solar',
 'ES1 2 CCGT',
 'ES1 2 coal',
 'ES1 2 offwind-ac',
 'ES1 2 onwind',
 'ES1 2 ror',
 'ES1 2 solar',
 'ES1 3 CCGT',
 'ES1 3 coal',
 'ES1 3 offwind-ac',
 'ES1 3 onwind',
 'ES1 3 solar',
 'ES1 4 CCGT',
 'ES1 4 coal',
 'ES1 4 offwind-ac',
 'ES1 4 onwind',
 'ES1 4 solar',
 'ES1 5 CCGT',
 'ES1 5 onwind',
 'ES1 5 solar',
 'ES1 6 CCGT',
 'ES1 6 offwind-ac',
 'ES1 6 onwind',
 'ES1 6 solar',
 'ES1 7 CCGT',
 'ES1 7 onwind',
 'ES1 7 solar',
 'ES1 8 CCGT',
 'ES1 8 offwind-ac',
 'ES1 8 onwind',
 'ES1 8 ror',
 'ES1 8 solar',
 'PT1 0 CCGT',
 'PT1 0 offwind-ac',
 'PT1 0 onwind',
 'PT1 0 ror',
 'PT1 0 solar',
 'PT1 1 CCGT',
 'PT1 1 offwind-ac',
 'PT1 1 onwind',
 'PT1 1 ror',
 'PT1 1 solar']

In [35]:
capacity_factors = input_data["capacity_factors"]

In [36]:
capacity_factors

ES1 0 offwind-ac  ES1 0 onwind  ES1 0 solar  ES1 1 offwind-ac  \
year week hour                                                                  
2025 1    0             0.087200      0.164996          0.0          0.409372   
          1             0.086428      0.147393          0.0          0.396427   
          2             0.080935      0.134339          0.0          0.448126   
          3             0.083694      0.136277          0.0          0.500791   
          4             0.108172      0.157959          0.0          0.627990   
...                          ...           ...          ...               ...   
2050 1    187           0.141622      0.116250          0.0          0.008258   
          188           0.108208      0.129085          0.0          0.000000   
          189           0.083249      0.134962          0.0          0.000000   
          190           0.095367      0.145156          0.0          0.000000   
          191           0.082320      0.136474          0.0          0.003845   

                ES1 1 onwind  ES1 1 solar  ES1 2 offwind-ac  ES1 2 onwind  \
year week hour                                                              
2025 1    0         0.316811          0.0          0.000000      0.112481   
          1         0.266069          0.0          0.000000      0.107711   
          2         0.244590          0.0          0.000000      0.110035   
          3         0.213173          0.0          0.014802      0.112983   
          4         0.213629          0.0          0.000000      0.129552   
...                      ...          ...               ...           ...   
2050 1    187       0.331133          0.0          0.000000      0.154126   
          188       0.330145          0.0          0.000000      0.182224   
          189       0.306421          0.0          0.000000      0.203858   
          190       0.355155          0.0          0.000000      0.222888   
          191       0.356562          0.0          0.000000      0.237029   

                ES1 2 ror  ES1 2 solar  ...  ES1 3 coal  ES1 1 CCGT  \
year week hour                          ...                           
2025 1    0      0.223631          0.0  ...         1.0         1.0   
          1      0.209677          0.0  ...         1.0         1.0   
          2      0.198885          0.0  ...         1.0         1.0   
          3      0.187313          0.0  ...         1.0         1.0   
          4      0.179705          0.0  ...         1.0         1.0   
...                   ...          ...  ...         ...         ...   
2050 1    187    0.300193          0.0  ...         1.0         1.0   
          188    0.288119          0.0  ...         1.0         1.0   
          189    0.279827          0.0  ...         1.0         1.0   
          190    0.273984          0.0  ...         1.0         1.0   
          191    0.269047          0.0  ...         1.0         1.0   

                ES1 8 CCGT  PT1 0 CCGT  ES1 2 CCGT  ES1 0 coal  PT1 1 CCGT  \
year week hour                                                               
2025 1    0            1.0         1.0         1.0         1.0         1.0   
          1            1.0         1.0         1.0         1.0         1.0   
          2            1.0         1.0         1.0         1.0         1.0   
          3            1.0         1.0         1.0         1.0         1.0   
          4            1.0         1.0         1.0         1.0         1.0   
...                    ...         ...         ...         ...         ...   
2050 1    187          1.0         1.0         1.0         1.0         1.0   
          188          1.0         1.0         1.0         1.0         1.0   
          189          1.0         1.0         1.0         1.0         1.0   
          190          1.0         1.0         1.0         1.0         1.0   
          191          1.0         1.0         1.0         1.0         1.0   

                ES1 4

In [40]:
# Initialize mapping
is_renewable = {}

In [44]:
import numpy as np

# Your list of generators:
G = [
    "ES1 0 CCGT",
    "ES1 0 coal",
    "ES1 0 offwind-ac",
    "ES1 0 onwind",
    "ES1 0 solar",
    "ES1 1 CCGT",
    "ES1 1 coal",
    "ES1 1 offwind-ac",
    "ES1 1 onwind",
    "ES1 1 solar",
    # ... (rest of your generators)
]

# Your capacity_factors DataFrame:
# Columns: generator names, MultiIndex (year, week, hour) as index

# Initialize mapping
is_renewable = {}


def _make_is_renewable_mapping(generator_list, capacity_factors_df):
    """
    Returns a mapping dictionary: {generator: True (renewable) or False (non-renewable)}
    A generator is considered non-renewable if its capacity factor is 1.0 for all time steps.
    Otherwise, it is considered renewable.

    Args:
        generator_list (list): List of generator names (str).
        capacity_factors_df (pd.DataFrame): DataFrame with generators as columns.

    Returns:
        dict: {generator: True/False}
    """
    mapping = {}
    for gen in generator_list:
        if gen not in capacity_factors_df.columns:
            mapping[gen] = np.nan  # or False, or raise an error
        else:
            vals = capacity_factors_df[gen].values
            if np.allclose(vals, 1.0, atol=1e-8):
                mapping[gen] = False  # Non-renewable
            else:
                mapping[gen] = True  # Renewable
    return mapping


is_renewable = _make_is_renewable_mapping(G, capacity_factors)
print(is_renewable)


# Print (or use) the mapping
for gen, renew in is_renewable.items():
    print(f"{gen}: {'renewable' if renew else 'non-renewable'}")
    print(is_renewable[gen])

{'ES1 0 CCGT': False, 'ES1 0 coal': False, 'ES1 0 offwind-ac': True, 'ES1 0 onwind': True, 'ES1 0 solar': True, 'ES1 1 CCGT': False, 'ES1 1 coal': False, 'ES1 1 offwind-ac': True, 'ES1 1 onwind': True, 'ES1 1 solar': True}
ES1 0 CCGT: non-renewable
False
ES1 0 coal: non-renewable
False
ES1 0 offwind-ac: renewable
True
ES1 0 onwind: renewable
True
ES1 0 solar: renewable
True
ES1 1 CCGT: non-renewable
False
ES1 1 coal: non-renewable
False
ES1 1 offwind-ac: renewable
True
ES1 1 onwind: renewable
True
ES1 1 solar: renewable
True
